In [25]:
import io
import torch
import whisper
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
from IPython.display import Audio as IPythonAudio, display
from itertools import islice

In [2]:
HF_TOKEN = "hf_mytoken"
WHISPER_MODEL = "medium"
N_SAMPLES = 10
SAMPLE_RATE = 16000
PAUSE_SECONDS = 0.5

In [3]:
dataset = load_dataset(
    "Sh1man/golos_opus",
    "crowd",
    split="test",
    streaming=True,
)

dataset = dataset.cast_column("opus", Audio(sampling_rate=SAMPLE_RATE))

audios = []
texts = []
for i, sample in enumerate(islice(dataset, N_SAMPLES)):
    audios.append(sample["opus"]["array"])

    if "json" in sample and isinstance(sample["json"], dict):
        text = sample["json"].get("text", "")
    else:
        text = sample.get("text", "")

    texts.append(text)
    print(f"  [{i+1}/{N_SAMPLES}] {text[:60]}...")

pause = np.zeros(int(PAUSE_SECONDS * SAMPLE_RATE), dtype=np.float32)
combined_audio = np.concatenate(
    [np.concatenate([a, pause]) for a in audios]
)
print(f"\nДлительность диалога: {len(combined_audio)/SAMPLE_RATE:.1f} с")

  [1/10] сбер включи дэйв томас джуниор...
  [2/10] заказать молоко уральское трех процентной жирности...
  [3/10] сколько будет стоить доставка заказа в отель...
  [4/10] набери фильм ватиканские записи...
  [5/10] афина повтори этот фрагмент еще раз...
  [6/10] найти на тв авто плюс...
  [7/10] кампус время работы...
  [8/10] включай на смотрешке егэ тв...
  [9/10] салют какое количество жителей страны не имеют доступа к инс...
  [10/10] афина как включить быстрый платеж...

Длительность диалога: 46.2 с


In [14]:
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN,
)
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))

combined_audio = combined_audio.astype(np.float32)
print(f"Тип аудио: {combined_audio.dtype}")
print(f"Длительность диалога: {len(combined_audio)/SAMPLE_RATE:.1f} с")

waveform = torch.from_numpy(combined_audio).unsqueeze(0)
audio_input = {"waveform": waveform, "sample_rate": SAMPLE_RATE}

diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN,
)
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))

with ProgressHook() as hook:
    diarization = diarization_pipeline(
        audio_input,
        hook=hook,
    )
    
diarization = diarization.speaker_diarization

Тип аудио: float32
Длительность диалога: 46.2 с


In [26]:
whisper_model = whisper.load_model(WHISPER_MODEL)

dialogue_lines = []
print("\n" + "="*60)
print("ТРАНСКРИПЦИЯ:")
print("="*60)

for turn, _, speaker in diarization.itertracks(yield_label=True):
    s = int(turn.start * SAMPLE_RATE)
    e = int(turn.end * SAMPLE_RATE)
    segment = combined_audio[s:e].astype(np.float32)

    if len(segment) < SAMPLE_RATE * 0.3:
        continue

    result = whisper_model.transcribe(segment, language="ru", fp16=False)
    text = result["text"].strip()

    if text:
        dialogue_lines.append((turn.start, speaker, text))
        print(f"[{turn.start:.1f}s] {speaker}: {text}")


ТРАНСКРИПЦИЯ:
[0.9s] SPEAKER_04: Сбер включи Дэйв Томас Джуню
[4.5s] SPEAKER_02: заказать молоко уральское 3% жирности.
[9.9s] SPEAKER_05: сколько будет стоить доставка заказа в отель.
[14.6s] SPEAKER_01: Небери фильм Ватиканские записи
[19.9s] SPEAKER_06: Афина повтори этот фрагмент еще раз
[24.3s] SPEAKER_04: найти на ТВ авто.
[29.0s] SPEAKER_03: Время работы.
[31.0s] SPEAKER_00: Включай насмотрежки EGTV.
[34.3s] SPEAKER_03: Салют, какое количество жителей страны не имеет доступа.
[39.3s] SPEAKER_03: институциональному банкингу.


In [22]:
import numpy as np
from IPython.display import Audio as IPythonAudio, display
import pandas as pd

speaker_intervals = {}
for turn, _, speaker in diarization.itertracks(yield_label=True):
    speaker_intervals.setdefault(speaker, []).append((turn.start, turn.end))

speakers = list(speaker_intervals.keys())
print(f"Обнаружено спикеров: {len(speakers)} → {speakers}")

tracks = {}
tracks_with_pauses = {}

for spk in speakers:
    segments = []
    for start, end in sorted(speaker_intervals[spk]):
        s, e = int(start * SAMPLE_RATE), int(end * SAMPLE_RATE)
        segments.append(combined_audio[s:e])

    if segments:
        tracks[spk] = np.concatenate(segments).astype(np.float32)
    else:
        tracks[spk] = np.zeros(0, dtype=np.float32)

    track_timed = np.zeros_like(combined_audio)
    for start, end in speaker_intervals[spk]:
        s, e = int(start * SAMPLE_RATE), int(end * SAMPLE_RATE)
        track_timed[s:e] = combined_audio[s:e]
    tracks_with_pauses[spk] = track_timed

    speech_sec = len(tracks[spk]) / SAMPLE_RATE
    print(f"  {spk}: {speech_sec:.1f} с чистой речи "
          f"({len(speaker_intervals[spk])} сегментов)")

summary = pd.DataFrame([
    {
        "Спикер": spk,
        "Сегментов": len(speaker_intervals[spk]),
        "Секунд речи": round(len(tracks[spk]) / SAMPLE_RATE, 2),
    }
    for spk in speakers
]).sort_values("Секунд речи", ascending=False)

display(summary)

Обнаружено спикеров: 7 → ['SPEAKER_04', 'SPEAKER_05', 'SPEAKER_02', 'SPEAKER_01', 'SPEAKER_06', 'SPEAKER_03', 'SPEAKER_00']
  SPEAKER_04: 5.2 с чистой речи (6 сегментов)
  SPEAKER_05: 3.4 с чистой речи (2 сегментов)
  SPEAKER_02: 4.0 с чистой речи (1 сегментов)
  SPEAKER_01: 2.9 с чистой речи (1 сегментов)
  SPEAKER_06: 2.5 с чистой речи (2 сегментов)
  SPEAKER_03: 8.1 с чистой речи (3 сегментов)
  SPEAKER_00: 2.2 с чистой речи (1 сегментов)


,Спикер,Сегментов,Секунд речи
5,SPEAKER_03,3,8.13
0,SPEAKER_04,6,5.23
2,SPEAKER_02,1,3.97
1,SPEAKER_05,2,3.38
3,SPEAKER_01,1,2.90
4,SPEAKER_06,2,2.55
6,SPEAKER_00,1,2.18


In [23]:
print("\n" + "="*60)
print("ПРОСЛУШИВАНИЕ:")
print("="*60)

print("Оригинальный диалог:")
display(IPythonAudio(combined_audio, rate=SAMPLE_RATE))

for spk in speakers:
    print(f"\n{spk}:")
    display(IPythonAudio(tracks[spk], rate=SAMPLE_RATE))


ПРОСЛУШИВАНИЕ:
Оригинальный диалог:



SPEAKER_04:



SPEAKER_05:



SPEAKER_02:



SPEAKER_01:



SPEAKER_06:



SPEAKER_03:



SPEAKER_00:
